# Tokenizers

Tokenizers are one of the core components of the NLP pipeline. They serve one purpose: to translate text into data that can be processed by the model.

Models can only process numbers, so tokenizers need to convert our text inputs to numerical data.

In NLP tasks, the data that is generally processed is raw text. Here is an example of such text:
`Jim Hension was a puppeteer`

However, models can only process numbers, so we need to find a way to convert the raw text to numbers. That is what tokenizers do, and there are a lot of ways to go about this.

The goal is to find the most meaningful representation - that is, the one that makes the most sense to the model - and, if possible, the smallest representation.

## Word-based
Generally very easy to set up and use with only a few rules, and it often yields decent results.

The goal is to split the raw text into words and find a numerical representaiton for each of them. There are different ways to split the text. For example, we could use whitespace to tokenize the text into words by applying Python's `split()` function:

In [1]:
split_text = "Jim Hension was a puppeteer".split()
print(split_text)

['Jim', 'Hension', 'was', 'a', 'puppeteer']


There are also variations of word tokenizers that have extra rules for punctuation.

With this kind of tokenizer, we can end up with some pretty large "vocabularies", where a vocabulary is defined by the total number of independent tokens that we have in our corpus.

Each word gets assigned an ID, starting from 0 and going up to the size of the vocabulary. The model uses these IDs to identify each word.

If we want to completely cover a language with a word-based tokenizer, we'll need to have an identifier for each word in the langauge, which will generate a huge amount tokens. For example, there are over 500,000 words in the English language, so to build a map from each word to an input ID we'd need to keep track of that many IDs (500,000). Furthermore, words like "dog" are represented differently from words like "dogs", and the model will initially have no way of knowing that "dog" and "dogs" are similar: it will identify the two words as unrelated. The same applies to similar words, like "run" and "running", which the model will not see as being similar initially.

Finally, we need a custom token to represent words that are not in our vocabulary. This is known as the "unknown" token, often represented ash "[UNK]" or "\<unk\>". It is generally a bad sign if you see that the tokenizer is producing a lot of these tokens, as it wasn't able to retrieve a sensible representation of a word and you're losing information along the way. The goal when crafting the vocabulary is to do it in such a way that the tokenizer tokenizes as few words as possible into the unknown token.

One way to reduce the amount of unkown tokens is to go one level deeper, using a character-based tokenizer.

## Character-based
Character-based tokenizers split the text into characters, rather than words. This has two primary benefits:
 * The vocabulary is much smaller.
 * There are much fewer out-of-vocabulary (unknown) tokens, since every word can be built from characters.

This approach isn't perfect either. Since the representation is now based on characters rather than words, one could argue that, intuitively, it's less meaningful: each character doesn't mean a lot on its own, whereas that is the case with words. However, this again differs according to the language; in Chinese, for example, each character carries more information than a character in a Latin language.

Another thing to consider is that we'll end up with a very large amount of tokens to be processed by our model: whereas a word would only be a single token with a word-based tokenizer, it can easily turn into 10 or more tokens when converted into characters.

To get the best of both worlds, we can use a third technique that combines the two approaches: `subword tokenization`.

## Subword tokenization

Subword toeknization algorithms rely on the principle that frequently used words should not be split into smaller subwords, but rare word should be dcomposed into meaningful subwords.

For instance, "annoyingly" might be considered a rare word and could be decomposed into "annoying" and "ly". These are both more liekly to appear more frequently as standalone subwords, while at the same time the meanging of "annoyingly' is kept by the compsite meaning of "annoying" and "ly".

Example showing how a subword tokenization algorithm would tokenize the sequence "Let's do tokenization!":
`Let's </w> | do</w> | token | iziation</w> | !</w>`

This approach is especially useful in agglutinative lagnauges such as Turkish, where you can form (almost) arbitrarily long complex words by stringing together subwords.

## Loading and Saving

Loading and saving tokenizers as simple as it is with models. Actually, it's based on the same two methods: `from_pretrained()` and `save_pretrained()`. 
These methods will load or save the algorithm use by the tokenizer as well as its vocabulary.

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
print(tokenizer)

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


The `AutoTokenizer` class will grab the proper tokenizer class in the library based on the checkpoint name, and can be used directly with any checkpoint.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Saving a tokenizer is identical to saving a model:

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

tokenizer.save_pretrained("/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers")

('/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers/tokenizer_config.json',
 '/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers/special_tokens_map.json',
 '/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers/vocab.txt',
 '/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers/added_tokens.json',
 '/Users/dimitaratanassov/Hugging Face/chapter2/tokenizers/tokenizer.json')

## Encoding
Translating text to numbers is known as `encoding`.
Encoding is done in a two-step process: the tokenization, followed by the conversion to input IDs.

As we've seen, the first step is to split the text into words (or parts of words, punctuation symbols, etc.), usually called `tokens`. There are multiple rules that can govern that process, which is why we need to instantiate the tokenizer using the name of the model, to make sure we use the same rules that were used when the model was pretrained.

The second step is to convert these tokens into numbers, so we can build a tensor out of them and feed them to the model. To do this, the tokenizer has a `vocabulary`, which is the part we download when we instantiate it with the `from_pretrained()` method. Again, we need to use the same vocabulary used when the model was pretrained.

*In practice, you should call the tokenizer directly on your inputs*

### Tokenization

The tokenization process is done by the `tokenize()` method of the tokenizer:

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

sequence = "A very boring example input string for a tokenizer"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['A', 'very', 'boring', 'example', 'input', 'string', 'for', 'a', 'token', '##izer']


The output of the `tokenize` method is a list of strings, or tokens:

`['A', 'very', 'boring', 'example', 'input', 'string', 'for', 'a', 'token', '##izer']`

This tokenizer is a subword tokenizer: it splits the words until it obtains tokens that can be represented by its vocabulary. That's the case here with `tokenizer`, which is split into two tokens: `token` and `##izer`.

### From tokens to input IDs

The conversion to input IDs is handled by the `convert_tokens_to_ids()` tokenizer method:

In [6]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[138, 1304, 12533, 1859, 7758, 5101, 1111, 170, 22559, 17260]


These outputs, once converted to the appropriate framework tensor, can then be used as inputs to a model as seen earlier in this chapter.

#### Another example

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

sequence = ["I’ve been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
tokens = tokenizer.tokenize(sequence)

print(tokens)

ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)



['I', '’', 've', 'been', 'waiting', 'for', 'a', 'Hu', '##gging', '##F', '##ace', 'course', 'my', 'whole', 'life', '.', 'I', 'hate', 'this', 'so', 'much', '!']
[146, 787, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 146, 4819, 1142, 1177, 1277, 106]


## Decoding

`Decoding` is going the other way around: from vocabulary indices, we want to get a string. This can be done with the `decode()` method as follows:

In [9]:
decoded_string = tokenizer.decode([138, 1304, 12533, 1859, 7758, 5101, 1111, 170, 22559, 17260])
print(decoded_string)

A very boring example input string for a tokenizer


Note that the `decode` method not only converts the indicies back to tokens, but also groups together the tokens that were part of the same words to produce a readable sentence. This behavior will be extremely useful when we use models that predict new text (either text generated from a prompt, or for sequence-to-sequence problems like translation or summarization).